# 7.17 Large Language Model Foundations

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook connects transformer mechanics to the core ideas behind autoregressive language modeling and modern LLM workflows.


## 7.17.1 Next-token prediction

### Why It Matters
A language model learns to predict the next token given the prefix. That objective is the backbone of many modern LLMs.


In [ ]:
import pandas as pd

text = "deep learning loves clean data"
tokens = text.split()
examples = []
for i in range(1, len(tokens)):
    examples.append({
        "prefix": " ".join(tokens[:i]),
        "next_token": tokens[i],
        "prefix_length": i,
    })

pd.DataFrame(examples)


### Interpretation
Each training example is a prefix and the next token that followed it in the corpus.


## 7.17.2 Pretraining versus fine-tuning

### Why It Matters
Pretraining learns a broad language representation, while fine-tuning adapts that model to a narrower downstream task.


In [ ]:
import torch
from torch import nn

encoder = nn.GRU(input_size=8, hidden_size=12, batch_first=True)
classification_head = nn.Linear(12, 2)
head_only_model = nn.Module()
head_only_model.encoder = encoder
head_only_model.head = classification_head
for param in head_only_model.encoder.parameters():
    param.requires_grad = False

full_tune_model = nn.GRU(input_size=8, hidden_size=12, batch_first=True)
full_head = nn.Linear(12, 2)

{
    "head_only_trainable_params": sum(p.numel() for p in list(head_only_model.encoder.parameters()) + list(head_only_model.head.parameters()) if p.requires_grad),
    "full_fine_tune_trainable_params": sum(p.numel() for p in list(full_tune_model.parameters()) + list(full_head.parameters()) if p.requires_grad),
}


### Interpretation
The distinction is about stage and objective, not about using a completely different kind of network.


## 7.17.3 Prompting as conditioning

### Why It Matters
A prompt conditions the distribution over next tokens by choosing the prefix the model sees.


In [ ]:
from collections import defaultdict, Counter

corpus = "the movie was great the movie was boring the market was calm the market was volatile".split()
transitions = defaultdict(Counter)
for left, right in zip(corpus[:-1], corpus[1:]):
    transitions[left][right] += 1

movie_distribution = transitions["movie"]
market_distribution = transitions["market"]
{
    "after_movie": dict(movie_distribution),
    "after_market": dict(market_distribution),
}


### Interpretation
Prompting is just controlled conditioning on context from the model’s point of view.


## 7.17.4 Context windows and tokenization

### Why It Matters
Language models only see a bounded number of tokens at once. Chunking a document into windows makes that limit concrete.


In [ ]:
import torch

text = "a course on neural networks needs concrete examples not placeholders".split()
vocab = {token: idx for idx, token in enumerate(sorted(set(text)))}
encoded = torch.tensor([vocab[token] for token in text])
window = 4
chunks = torch.stack([encoded[i:i+window] for i in range(len(encoded) - window + 1)])

{
    "vocab_size": len(vocab),
    "encoded_length": len(encoded),
    "windowed_tensor_shape": list(chunks.shape),
    "first_window": chunks[0].tolist(),
}


### Interpretation
A context window is a hard budget on how much of the previous text can influence the next prediction.


## 7.17.5 Sampling temperature and top-k intuition

### Why It Matters
Generation is not just taking the most likely token every time. Temperature rescales logits, and top-k restricts the candidate set.


In [ ]:
import torch

logits = torch.tensor([2.0, 1.0, 0.2, -0.5])
def probs(temp):
    return torch.softmax(logits / temp, dim=0).round(decimals=3).tolist()
topk = torch.topk(logits, k=2).indices.tolist()
{"temperature_0.5": probs(0.5), "temperature_1.5": probs(1.5), "top_2_indices": topk}


### Interpretation
Lower temperature sharpens the distribution; higher temperature makes it flatter.


## 7.17.6 A tiny autoregressive workflow

### Why It Matters
This final subsection trains a very small decoder-style model on character sequences so the next-token objective becomes executable.


In [ ]:
import torch
from torch import nn

text = "hello neural world"
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
encoded = [stoi[c] for c in text]
X = torch.tensor([encoded[i:i+4] for i in range(len(encoded) - 4)], dtype=torch.long)
y = torch.tensor([encoded[i+4] for i in range(len(encoded) - 4)], dtype=torch.long)

class TinyDecoder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.GRU(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, vocab_size)
    def forward(self, x):
        emb = self.embedding(x)
        out, hidden = self.rnn(emb)
        return self.head(hidden[-1])

model = TinyDecoder(len(chars), 12)
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()
for _ in range(120):
    logits = model(X)
    loss = loss_fn(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
{"final_loss": round(float(loss.item()), 4), "vocab_size": len(chars), "training_examples": len(X)}


### Interpretation
This is not a large model, but it trains under the same next-token logic used by much larger systems.
